# 💻 Chapter 17: Log Probabilities & Numerical Stability
*Track 2 — Developers | Tier 2*

> **🎬 Hook:** Multiply 1,000 small probabilities together and Python returns 0.0. Not because the answer is zero — because floating point underflowed. Here's the fix every NLP engineer must know.

**🎯 Objectives:** Understand float underflow, the log-probability trick, and the log-sum-exp algorithm.

## 📖 Section 1 — Concept Review

### The Underflow Problem
```python
probs = [0.1] * 1000
product = 1.0
for p in probs: product *= p
print(product)  # → 0.0  ❌ (true answer ≈ 10^-1000)
```

### The Fix: Work in Log Space
$$\log(p_1 \cdot p_2 \cdots p_n) = \log p_1 + \log p_2 + \cdots + \log p_n$$

Multiplication → Addition in log space. No underflow!

### Log-Sum-Exp Trick
Computing log(a + b) from log(a) and log(b):
$$\log(e^a + e^b) = a + \log(1 + e^{b-a}) \quad \text{(take max a)}$$

Or: `scipy.special.logsumexp(log_probs)` — always use this!

### Where This Appears
- Naive Bayes classifier: P(class|features) = ∏ P(feature|class)
- Language models: P(sentence) = ∏ P(word|context)
- Viterbi algorithm, HMMs, CRFs
- Any model with many multiplication terms

In [ ]:
import numpy as np
from scipy.special import logsumexp
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

# ── Underflow Demo ──
print("🚨 Underflow Problem")
n_probs = [10, 100, 500, 1000, 2000]
for n in n_probs:
    probs = [0.1] * n
    naive = 1.0
    for p in probs: naive *= p
    log_result = sum(np.log(p) for p in probs)
    print(f"  n={n:>4}: naive product = {naive:.2e}, log-space = {log_result:.2f}, recovered = {np.exp(log_result):.2e}")

print()
print("✅ Solution: Always work in log space, convert back only at the end!")

In [ ]:
# ── Log-Sum-Exp Trick ──
print("🔢 Log-Sum-Exp Trick")
print()

# Problem: compute log(e^a + e^b) without overflow/underflow
a, b = 1000, 1001

# Naive: e^1000 overflows!
try:
    naive = np.log(np.exp(a) + np.exp(b))
    print(f"  Naive result: {naive}")
except:
    print("  Naive: OVERFLOW!")

# Log-sum-exp: numerically stable
# log(e^a + e^b) = b + log(1 + e^(a-b))  [using max=b]
stable = b + np.log(1 + np.exp(a - b))
scipy_lse = logsumexp([a, b])

print(f"  Manual log-sum-exp: {stable:.4f}")
print(f"  scipy.special.logsumexp: {scipy_lse:.4f}")
print(f"  True value: {np.log(np.exp(a-b) + 1) + b:.4f}")
print()

# Softmax in log space
def stable_softmax(logits):
    return np.exp(logits - logsumexp(logits))

logits = np.array([1.0, 2.0, 3.0, 1.0])
probs = stable_softmax(logits)
print(f"Logits: {logits}")
print(f"Softmax: {probs}")
print(f"Sum = {probs.sum():.6f}")

In [ ]:
# ── Naive Bayes Classifier in Log Space ──
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer

# Simple Naive Bayes from scratch (in log space!)
class LogSpaceNaiveBayes:
    def fit(self, X, y):
        self.classes_ = np.unique(y)
        n_features = X.shape[1]
        self.log_priors_ = {}
        self.log_likelihoods_ = {}  # log P(feature | class)

        for c in self.classes_:
            mask = y == c
            X_c = X[mask]
            self.log_priors_[c] = np.log(mask.sum() / len(y))
            # Laplace smoothing in log space
            counts = X_c.sum(axis=0) + 1  # +1 smoothing
            self.log_likelihoods_[c] = np.log(counts / counts.sum())

    def predict_log_proba(self, X):
        log_probs = {}
        for c in self.classes_:
            # log P(c|x) ∝ log P(c) + Σ log P(xi|c)
            log_probs[c] = self.log_priors_[c] + X.dot(self.log_likelihoods_[c])
        return log_probs

    def predict(self, X):
        log_probs = self.predict_log_proba(X)
        predictions = np.array([
            max(self.classes_, key=lambda c: log_probs[c][i])
            for i in range(X.shape[0])
        ])
        return predictions

# Quick demo on synthetic data
rng = np.random.default_rng(42)
n = 200
# Simulate bag-of-words features
X_spam = np.column_stack([rng.poisson(5, n//2), rng.poisson(1, n//2), rng.poisson(2, n//2)])
X_ham  = np.column_stack([rng.poisson(1, n//2), rng.poisson(4, n//2), rng.poisson(3, n//2)])
X = np.vstack([X_spam, X_ham])
y = np.array(['spam']*100 + ['ham']*100)

nb = LogSpaceNaiveBayes()
nb.fit(X, y)
predictions = nb.predict(X)
accuracy = (predictions == y).mean()
print(f"Naive Bayes accuracy: {accuracy:.2%}")
print("Key: all computations done in log space, no underflow!")

## ✏️ Section 6 — Coding Challenges

**Challenge 1:** Compute log P(sentence) for a language model.
Sentence: "the cat sat on the mat". Unigram probs: {the:0.05, cat:0.01, sat:0.008, on:0.04, mat:0.003}.

**Challenge 2:** Implement `log_normalize(log_probs)` that normalizes a list of log probabilities.
Hint: use logsumexp.

**Challenge 3:** Why is cross-entropy loss in neural networks actually log-probability?
<details><summary>Solutions</summary>

**C1:** log P = Σ log(p_word). Sum the log probabilities.

**C2:** `log_probs - logsumexp(log_probs)`

**C3:** Cross-entropy = -E[log P(y|x)] = -log P(correct class). Minimizing CE = maximizing log-likelihood = MLE.
</details>

In [ ]:
# C1
sentence = "the cat sat on the mat".split()
unigram_probs = {'the':0.05,'cat':0.01,'sat':0.008,'on':0.04,'mat':0.003}
log_probs = [np.log(unigram_probs[w]) for w in sentence]
print(f"Log P(sentence) = {sum(log_probs):.4f}")
print(f"P(sentence) = {np.exp(sum(log_probs)):.2e}")

# C2
def log_normalize(log_probs):
    return np.array(log_probs) - logsumexp(log_probs)
lp = [-1.0, -2.0, -0.5, -3.0]
normalized = log_normalize(lp)
print(f"
Log probs: {lp}")
print(f"Normalized: {normalized}")
print(f"Exp(normalized): {np.exp(normalized)} (sum={np.exp(normalized).sum():.4f})")

## 🎯 Recap
1. Multiplying many small probabilities causes underflow — always use log space.
2. Log-sum-exp trick enables numerically stable softmax, marginalization, and normalization.
3. Log-probabilities are fundamental to NLP, HMMs, and probabilistic graphical models.

**Next:** [Chapter 18 — Probability in APIs & Rate Limiting]